In [15]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------------------------------------------------------
# Adjust import path to where these functions live in your package
# -----------------------------------------------------------------------------
from stac2cube import export_stac, export_to_cogs


def launch_stac2cube_slicer(default_var="Spectral_Temporal_Stack"):
    """
    Interactive slicer for stac2cube-like NetCDF outputs in Jupyter.

    Export logic:
    - NetCDF -> export_stac(...)
    - COGs   -> export_to_cogs(...)

    Notes
    -----
    - Opens NetCDF lazily (xr.open_dataset).
    - Band/time selections are optional (empty = keep all).
    - No spatial slicing (time/band-focused).
    """

    # -------------------------
    # Internal state
    # -------------------------
    state = {
        "ds": None,            # open Dataset handle
        "da": None,            # selected DataArray variable
        "loaded_path": None,   # input path
    }

    # -------------------------
    # Helpers
    # -------------------------
    def _safe_close_dataset():
        ds = state.get("ds", None)
        if ds is not None:
            try:
                ds.close()
            except Exception:
                pass
        state["ds"] = None
        state["da"] = None
        state["loaded_path"] = None

    def _fmt_time_list(time_values, max_show=8):
        if len(time_values) == 0:
            return "[]"
        strs = [str(pd.to_datetime(t)) for t in time_values]
        if len(strs) <= max_show:
            return strs
        return strs[:max_show] + [f"... ({len(strs)-max_show} more)"]

    def _summarize_dataarray(da: xr.DataArray):
        lines = []
        lines.append(f"Name: {da.name}")
        lines.append(f"Dims: {dict(da.sizes)}")
        lines.append(f"dtype: {da.dtype}")

        try:
            nbytes_mb = da.nbytes / (1024 ** 2)
            lines.append(f"Approx size in memory (if fully loaded): {nbytes_mb:.2f} MB")
        except Exception:
            pass

        if "time" in da.coords:
            try:
                tvals = da["time"].values
                lines.append(f"time ({len(tvals)}): {_fmt_time_list(tvals)}")
            except Exception:
                lines.append("time: <unavailable>")

        if "band" in da.coords:
            try:
                bvals = da["band"].values.tolist()
                lines.append(f"band ({len(bvals)}): {bvals}")
            except Exception:
                lines.append("band: <unavailable>")

        if da.attrs:
            lines.append("attrs:")
            for k, v in da.attrs.items():
                v_str = str(v)
                if len(v_str) > 120:
                    v_str = v_str[:117] + "..."
                lines.append(f"  - {k}: {v_str}")
        else:
            lines.append("attrs: {}")

        return "\n".join(lines)

    def _get_selected_da():
        da = state["da"]
        if da is None:
            raise ValueError("No cube loaded yet.")

        sub = da

        # Band selection
        if "band" in sub.coords and len(bands_select.value) > 0:
            sub = sub.sel(band=list(bands_select.value))

        # Time selection
        if "time" in sub.coords and len(times_select.value) > 0:
            times = [np.datetime64(t) for t in times_select.value]
            sub = sub.sel(time=times)

        return sub

    def _auto_output_target(input_path, export_kind):
        if export_kind == "netcdf":
            p = Path(input_path)
            stem = p.stem
            parent = p.parent
            return str(parent / f"{stem}_sliced.nc")

        elif export_kind == "cogs":
            # requested default folder
            return r"./results/cogs"

        else:
            raise ValueError(f"Unsupported export kind: {export_kind}")

    def _update_output_field_ui():
        if out_format.value == "netcdf":
            out_path.description = "Output file:"
            out_path.placeholder = "./results/test_sliced.nc"
        elif out_format.value == "cogs":
            out_path.description = "Output dir:"
            out_path.placeholder = r"./results/cogs"

    def _export_with_stac2cube(sub: xr.DataArray, export_kind: str, target_path: str):
        """
        Uses your exact exporter signatures.
        """
        if export_kind == "netcdf":
            export_stac(
                stac=sub,
                output=target_path,
                var_name=(sub.name or (var_name.value.strip() if var_name.value.strip() else None)),
            )

        elif export_kind == "cogs":
            export_to_cogs(
                stac=sub,
                output_dir=target_path,
                prefix="",
                dtype="float32",
            )
        else:
            raise ValueError(f"Unsupported export kind: {export_kind}")

    # -------------------------
    # Widgets
    # -------------------------
    in_path = widgets.Text(
        value="",
        description="Input .nc:",
        placeholder="./results/test.nc",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "100px"},
    )

    var_name = widgets.Text(
        value=default_var,
        description="Variable:",
        layout=widgets.Layout(width="45%"),
        style={"description_width": "100px"},
    )

    out_format = widgets.Dropdown(
        options=[
            ("NetCDF", "netcdf"),
            ("Cloud Optimized Geotiffs (select folder)", "cogs"),
        ],
        value="netcdf",
        description="Export:",
        layout=widgets.Layout(width="55%"),
        style={"description_width": "70px"},
    )

    out_path = widgets.Text(
        value="",
        description="Output file:",
        placeholder="./results/test_sliced.nc",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "100px"},
    )

    load_btn = widgets.Button(description="Load cube", button_style="primary", icon="folder-open")
    preview_btn = widgets.Button(description="Preview selection", icon="eye")
    export_btn = widgets.Button(description="Slice sub data cube", button_style="success", icon="save")

    bands_select = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Bands",
        rows=8,
        layout=widgets.Layout(width="48%", height="220px"),
        style={"description_width": "70px"},
    )

    times_select = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Times",
        rows=8,
        layout=widgets.Layout(width="48%", height="220px"),
        style={"description_width": "70px"},
    )

    select_all_bands_btn = widgets.Button(description="All bands", layout=widgets.Layout(width="120px"))
    clear_bands_btn = widgets.Button(description="Clear bands", layout=widgets.Layout(width="120px"))
    select_all_times_btn = widgets.Button(description="All times", layout=widgets.Layout(width="120px"))
    clear_times_btn = widgets.Button(description="Clear times", layout=widgets.Layout(width="120px"))

    info_out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "6px"})
    preview_out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "6px"})
    msg_out = widgets.Output()

    # Disable until load
    for w in [bands_select, times_select, preview_btn, export_btn]:
        w.disabled = True

    _update_output_field_ui()

    # -------------------------
    # Callbacks
    # -------------------------
    def on_load_clicked(_):
        with msg_out:
            clear_output()
            print("Loading...")

        with info_out:
            clear_output()

        with preview_out:
            clear_output()

        try:
            path = in_path.value.strip()
            if not path:
                raise ValueError("Please provide an input NetCDF path.")
            if not os.path.exists(path):
                raise FileNotFoundError(f"File not found: {path}")

            _safe_close_dataset()

            ds = xr.open_dataset(path)  # lazy open
            var = var_name.value.strip()
            if var not in ds.data_vars:
                available = list(ds.data_vars)
                ds.close()
                raise KeyError(f"Variable '{var}' not found. Available data_vars: {available}")

            da = ds[var]

            state["ds"] = ds
            state["da"] = da
            state["loaded_path"] = path

            # Populate band options
            if "band" in da.coords:
                band_vals = [str(b) for b in da["band"].values.tolist()]
                bands_select.options = band_vals
                bands_select.value = ()
            else:
                bands_select.options = []
                bands_select.value = ()

            # Populate time options
            if "time" in da.coords:
                time_vals = pd.to_datetime(da["time"].values)
                time_strs = [str(t.to_datetime64()) for t in time_vals]
                times_select.options = time_strs
                times_select.value = ()
            else:
                times_select.options = []
                times_select.value = ()

            for w in [bands_select, times_select, preview_btn, export_btn]:
                w.disabled = False

            current = out_path.value.strip()
            if not current:
                out_path.value = _auto_output_target(path, out_format.value)
            else:
                # refresh only obvious auto-generated targets
                if "_sliced" in Path(current).name or current in [r"./results/cogs", "results/cogs"]:
                    out_path.value = _auto_output_target(path, out_format.value)

            with info_out:
                clear_output()
                print("✅ Cube loaded (lazy)")
                print(_summarize_dataarray(da))

            with msg_out:
                clear_output()
                print("Loaded successfully.")

        except Exception as e:
            with msg_out:
                clear_output()
                print(f"❌ {type(e).__name__}: {e}")

    def on_preview_clicked(_):
        with preview_out:
            clear_output()
        with msg_out:
            clear_output()

        try:
            sub = _get_selected_da()

            with preview_out:
                print("Selection summary")
                print("-----------------")
                print(_summarize_dataarray(sub))
                print("\nDataArray repr:")
                display(sub)

            with msg_out:
                print("Preview ready.")

        except Exception as e:
            with msg_out:
                clear_output()
                print(f"❌ {type(e).__name__}: {e}")

    def on_export_clicked(_):
        with msg_out:
            clear_output()
            print("Exporting...")

        try:
            sub = _get_selected_da()

            target = out_path.value.strip()
            if not target:
                if state["loaded_path"] is None:
                    raise ValueError("No loaded file path found to build auto output target.")
                target = _auto_output_target(state["loaded_path"], out_format.value)
                out_path.value = target

            if out_format.value == "netcdf":
                if not target.lower().endswith(".nc"):
                    target = target + ".nc"
                    out_path.value = target
                Path(target).parent.mkdir(parents=True, exist_ok=True)

            elif out_format.value == "cogs":
                # COG export expects folder output
                Path(target).mkdir(parents=True, exist_ok=True)

                # export_to_cogs requires a DataArray with "band" dim
                if "band" not in sub.dims:
                    raise ValueError(
                        "COG export requires a 'band' dimension."
                    )

            # IMPORTANT: always use package exporters
            _export_with_stac2cube(sub, out_format.value, target)

            with msg_out:
                clear_output()
                print("✅ Export finished")
                print(f"Format: {out_format.value}")
                print(f"Saved to: {target}")
                print(f"Export dims: {dict(sub.sizes)}")

        except Exception as e:
            with msg_out:
                clear_output()
                print(f"❌ {type(e).__name__}: {e}")

    def on_out_format_change(change):
        if change["name"] == "value" and change["new"] != change["old"]:
            _update_output_field_ui()

            # set sensible default immediately when switching to COGs
            if out_format.value == "cogs" and not out_path.value.strip():
                out_path.value = r"./results/cogs"

            if state["loaded_path"]:
                current = out_path.value.strip()
                auto_like = (
                    (not current)
                    or ("_sliced" in Path(current).name)
                    or (current in [r"./results/cogs", "results/cogs"])
                )
                if auto_like:
                    out_path.value = _auto_output_target(state["loaded_path"], out_format.value)

    def on_select_all_bands(_):
        bands_select.value = tuple(bands_select.options)

    def on_clear_bands(_):
        bands_select.value = ()

    def on_select_all_times(_):
        times_select.value = tuple(times_select.options)

    def on_clear_times(_):
        times_select.value = ()

    # Wire callbacks
    load_btn.on_click(on_load_clicked)
    preview_btn.on_click(on_preview_clicked)
    export_btn.on_click(on_export_clicked)
    out_format.observe(on_out_format_change, names="value")
    select_all_bands_btn.on_click(on_select_all_bands)
    clear_bands_btn.on_click(on_clear_bands)
    select_all_times_btn.on_click(on_select_all_times)
    clear_times_btn.on_click(on_clear_times)

    # -------------------------
    # Layout
    # -------------------------
    header = widgets.HTML("<h3 style='margin:0;'>stac2cube Interactive Cube Slicer</h3>")

    row1 = widgets.HBox([var_name, out_format])
    row2 = widgets.HBox([load_btn, preview_btn, export_btn])

    bands_box = widgets.VBox(
        [
            bands_select,
            widgets.HBox([select_all_bands_btn, clear_bands_btn]),
        ],
        layout=widgets.Layout(width="49%"),
    )

    times_box = widgets.VBox(
        [
            times_select,
            widgets.HBox([select_all_times_btn, clear_times_btn]),
        ],
        layout=widgets.Layout(width="49%"),
    )

    selectors = widgets.HBox(
        [bands_box, times_box],
        layout=widgets.Layout(justify_content="space-between")
    )

    ui = widgets.VBox(
        [
            header,
            in_path,
            row1,
            out_path,
            row2,
            widgets.HTML("<b>Loaded cube summary</b>"),
            info_out,
            widgets.HTML("<b>Band / time selection</b>"),
            selectors,
            widgets.HTML("<b>Preview</b>"),
            preview_out,
            widgets.HTML("<b>Status</b>"),
            msg_out,
        ]
    )

    display(ui)
    return ui


In [ ]:
ui = launch_stac2cube_slicer()

In [23]:
import ast
import os
import re
from pathlib import Path

import xarray as xr
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

from stac2cube import missions, get_stac_layers, export_stac, export_to_cogs


# -------------------------------------------------------------------------
# Parameter help (extracted/adapted from your HELP_MD)
# -------------------------------------------------------------------------
PARAM_HELP_HTML = {
    "daterange_mode": """
    <b>Date Range Mode</b><br>
    Choose how <code>daterange</code> is interpreted:<br><br>
    <b>1) Standard (single window)</b><br>
    <code>["YYYY-MM-DD", "YYYY-MM-DD"]</code><br><br>
    e.g., <code>["2024-04-01", "2024-04-07"]</code><br><br>
    <b>2) Seasonal (repeat across years)</b><br>
    <code>["MM-DD", "MM-DD"]</code><br>
    Example: vegetation season <code>["04-01", "10-31"]</code><br><br>
    <b>3) Seasonal + year control</b><br>
    <code>{"season": ["MM-DD", "MM-DD"], "years": "all"}</code><br>
    <code>{"season": ["MM-DD", "MM-DD"], "years": [2019, 2020, 2021]}</code><br>
    <code>{"season": ["MM-DD", "MM-DD"], "years": "2018-2024"}</code>
    """,

    "polygon": """
    <b>1) Path to polygon file</b><br>
    Polygon formats: <code>gpkg</code>, <code>geojson</code>, <code>kml</code>, <code>kmz</code>, <code>shp</code>.<br>
    Polygons can be geographic (WGS84) or projected (e.g., UTM).<br>
    <b>2) BBOX List</b><br>
    Can also be a WGS84 bbox list: <code>[xmin, ymin, xmax, ymax]</code> (never projected coords!). useful tool: <code>http://bboxfinder.com/</code><br>
    <b>Note:</b> If you have multiple features, only the first feature is used.
    """,

    "clip_raster": """
    <b>clip_raster</b><br>
    <b>True</b>: clip raster to polygon area.<br>
    <b>False</b>: keep polygon bounding box extent.<br><br>
    Keep <b>False</b> if you plan co-registration (bbox shape works best).<br>
    After co-registration you can clip using <code>clip_stac()</code>.
    """,

    "max_cc": """
    <b>max_cc</b><br>
    Maximum cloud coverage (%) from STAC metadata.<br>
    Keeping <code>100</code> is recommended for maximum availability.
    """,

    "cloud_masking": """
    <b>cloud_masking</b><br>
    Uses Scene Classification Layer masking (not s2cloudless threshold masking).<br><br>
    Keep <b>False</b> if you want to generate a cloud mask cube and choose your own threshold later.<br>
    Set <b>True</b> for quick/rough masking (e.g., large areas).
    """,

    "stats": """
    <b>stats</b><br>
    If empty/None: no stats cubes.<br>
    Creates additional data variables with requested statistics.<br><br>
    Examples:
    <ul style="margin:4px 0 0 18px; padding:0;">
        <li><code>mean_timeseries</code></li>
        <li><code>mean_monthly</code></li>
        <li><code>mean_annual</code></li>
        <li><code>mean_all</code> = timeseries + monthly + annual</li>
    </ul>
    Disabled when <code>aggregator</code> is not None.
    """,

    "aggregator": """
    <b>aggregator</b><br>
    Generates a single aggregated scene along time for each selected band/index.<br>
    Typically <code>mean</code> or <code>median</code>.<br><br>
    If <b>None</b>: no aggregation.<br>
    Setting an aggregator disables <code>stats</code>.
    """,

    "output": """
    <b>Output</b><br>
    <b>1) Quick Result, no Export</b> → returns lazy array, select this to check the data cube before exporting!<br>
    <b>2) NetCDF + Output file set</b> → generates single file multispectral + multidate data cube<br>
    <b>3) COGs + Output directory set</b> → generates multispectral GeoTiffs per each selected date<br><br>
    <b>Note:</b> You can generate lazily first, inspect the result, then switch export mode and export later.
    """,
}


def _make_help_toggle(help_key: str):
    btn = widgets.Button(
        description="?",
        tooltip="Show help",
        layout=widgets.Layout(width="22px", min_width="22px", height="22px", padding="0px"),
    )
    btn.style.button_color = "#2563eb"  # blue
    try:
        btn.style.text_color = "white"
    except Exception:
        pass
    try:
        btn.style.font_weight = "bold"
    except Exception:
        pass

    help_html = widgets.HTML(
        value=f"""
        <div style="
            border:1px solid #dbeafe;
            border-radius:8px;
            padding:8px 10px;
            margin:2px 0 8px 0;
            line-height:1.35;
            font-size:12.5px;
            background:#eff6ff;
        ">
            {PARAM_HELP_HTML.get(help_key, "No help available.")}
        </div>
        """,
        layout=widgets.Layout(display="none"),
    )

    def _toggle(_):
        help_html.layout.display = "" if help_html.layout.display == "none" else "none"

    btn.on_click(_toggle)
    return btn, help_html


def _with_help_left(widget, help_key: str, label_text: str = None):
    """
    Clean layout:
    - first row: Label + ? (inline, no forced width)
    - second row: widget full width
    - third row: collapsible help box
    """
    btn, help_html = _make_help_toggle(help_key)

    if label_text is None and hasattr(widget, "description"):
        label_text = widget.description or ""
    label_text = (label_text or "").strip()
    if label_text and not label_text.endswith(":"):
        label_text = f"{label_text}:"

    # Hide built-in widget description to avoid duplicate labels
    if hasattr(widget, "description"):
        try:
            widget.description = ""
        except Exception:
            pass
    if hasattr(widget, "style"):
        try:
            widget.style.description_width = "0px"
        except Exception:
            pass

    label_html = widgets.HTML(
        value=f"""
        <div style="
            font-weight:500;
            line-height:1.2;
            white-space:nowrap;
            margin:0;
            padding:0;
        ">{label_text}</div>
        """,
        layout=widgets.Layout(width="auto"),
    )

    label_row = widgets.HBox(
        [label_html, btn],
        layout=widgets.Layout(
            width="auto",
            align_items="center",
            justify_content="flex-start",
            gap="4px",
        ),
    )

    widget_box = widgets.Box([widget], layout=widgets.Layout(width="100%"))

    return widgets.VBox(
        [label_row, widget_box, help_html],
        layout=widgets.Layout(width="100%")
    )


def _stacked_field(widget, label_text: str = None):
    """
    Same visual style as _with_help_left(), but without question-mark help.
    - first row: label
    - second row: widget (full width)
    """
    if label_text is None and hasattr(widget, "description"):
        label_text = widget.description or ""
    label_text = (label_text or "").strip()
    if label_text and not label_text.endswith(":"):
        label_text = f"{label_text}:"

    if hasattr(widget, "description"):
        try:
            widget.description = ""
        except Exception:
            pass
    if hasattr(widget, "style"):
        try:
            widget.style.description_width = "0px"
        except Exception:
            pass

    label_html = widgets.HTML(
        value=f"""
        <div style="
            font-weight:500;
            line-height:1.2;
            white-space:nowrap;
            margin:0;
            padding:0;
        ">{label_text}</div>
        """,
        layout=widgets.Layout(width="auto"),
    )

    widget_box = widgets.Box([widget], layout=widgets.Layout(width="100%"))

    return widgets.VBox(
        [label_html, widget_box],
        layout=widgets.Layout(width="100%")
    )


def launch_stac2cube_generator_form(missions_func=missions):
    """
    Mission-driven stac2cube GUI for Jupyter.

    Supports:
    - lazy generation (default)
    - direct NetCDF export via get_stac_layers(output=...)
    - deferred NetCDF/COG export from current result
    """

    # -------------------------------------------------------------------------
    # Helpers
    # -------------------------------------------------------------------------
    def _to_list_or_empty(v):
        return v if isinstance(v, list) else []

    def _is_supported(v):
        return v is not False and v is not None

    def _pretty_mission_label(name: str):
        custom = {
            "sentinel_2_l2a": "Sentinel 2 L2A",
            "sentinel_2_l1c": "Sentinel 2 L1C",
            "sentinel_1_rtc": "Sentinel 1 RTC",
            "landsat_c2_l2": "Landsat Collection 2 Level 2",
        }
        return custom.get(name, name.replace("_", " ").title())

    def _bool_dropdown_from_metadata(value, default=False):
        if value is False:
            return {
                "options": [("Not available", None)],
                "value": None,
                "disabled": True,
            }

        options = [("False", False), ("True", True)]
        if isinstance(value, list):
            bools = [v for v in [False, True] if v in value]
            options = [(str(v), v) for v in bools] if bools else [("False", False), ("True", True)]

        return {
            "options": options,
            "value": default if any(v == default for _, v in options) else options[0][1],
            "disabled": False,
        }

    def _band_resolution_map(mission_name: str):
        if mission_name in {"sentinel_2_l2a", "sentinel_2_l1c"}:
            return {
                "coastal": "60m",
                "blue": "10m",
                "green": "10m",
                "red": "10m",
                "rededge1": "20m",
                "rededge2": "20m",
                "rededge3": "20m",
                "nir": "10m",
                "nir08": "20m",
                "nir09": "60m",
                "cirrus": "60m",
                "swir16": "20m",
                "swir22": "20m",
            }
        elif mission_name == "sentinel_1_rtc":
            return {"vh": "10m", "vv": "10m"}
        elif mission_name == "landsat_c2_l2":
            return {
                "coastal": "30m",
                "blue": "30m",
                "green": "30m",
                "red": "30m",
                "nir": "30m",
                "swir1": "30m",
                "swir2": "30m",
                "thermal": "30m",
            }
        return {}

    def _index_fullname_map(mission_name: str):
        common = {
            "ndvi": "Normalized Difference Vegetation Index",
            "ndwi": "Normalized Difference Water Index",
            "savi": "Soil Adjusted Vegetation Index",
            "ndmi": "Normalized Difference Moisture Index",
            "nbr": "Normalized Burn Ratio",
            "mndwi": "Modified Normalized Difference Water Index",
            "ndbi": "Normalized Difference Built-up Index",
            "evi": "Enhanced Vegetation Index",
            "ndre1": "Normalized Difference Red Edge Index",
            "ndsi": "Normalized Difference Snow Index",
        }

        radar = {
            "vh/vv": "VH/VV Ratio",
            "vv/vh": "VV/VH Ratio",
            "rvi": "Radar Vegetation Index",
        }

        if mission_name == "sentinel_1_rtc":
            return radar
        return common

    def _index_options_with_fullname(mission_name: str, index_list):
        name_map = _index_fullname_map(mission_name)
        options = []
        for idx in index_list:
            full = name_map.get(idx)
            label = f"{idx} ({full})" if full else str(idx)
            options.append((label, idx))
        return options

    def _band_options_with_resolution(mission_name: str, band_list):
        """
        UI labels show native resolution and are sorted by resolution where possible.
        Example for Sentinel-2:
        10m bands first (blue, green, red, nir), then 20m, then 60m.
        Selected values remain raw band names.
        """
        res_map = _band_resolution_map(mission_name)

        indexed = list(enumerate(band_list))  # preserve original order within same resolution (stable)

        def _res_rank(item):
            idx, band = item
            res = res_map.get(band, "")
            m = re.match(r"^(\d+)m$", str(res))
            if m:
                return (int(m.group(1)), idx)
            # unknown/no resolution goes last, preserving original order
            return (9999, idx)

        indexed_sorted = sorted(indexed, key=_res_rank)

        options = []
        for _, b in indexed_sorted:
            res = res_map.get(b)
            label = f"{b} ({res})" if res else str(b)
            options.append((label, b))
        return options


    def _daterange_mode_placeholder(mode_value: str):
        if mode_value == "standard":
            return '["2024-04-01", "2024-04-07"]'
        elif mode_value == "seasonal":
            return '["04-01", "10-31"]'
        elif mode_value == "seasonal_years":
            return '{"season": ["04-01", "10-31"], "years": [2019, 2020, 2021]}'
        return '["2024-04-01", "2024-04-07"]'

    def _auto_netcdf_suggestion_from_polygon():
        """
        Build a default NetCDF output path from polygon input.
        Examples:
        - ./polygons/test.gpkg -> ./results/test.nc
        - [xmin, ymin, xmax, ymax] -> ./results/bbox.nc
        """
        raw = (polygon_w.value or "").strip()

        # bbox input -> generic filename
        if raw.startswith("[") or raw.startswith("("):
            stem = "bbox"
        elif raw:
            # path-like input
            try:
                stem = Path(raw).stem
            except Exception:
                stem = "test"
        else:
            stem = "test"

        # sanitize a little for filenames
        stem = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._-") or "test"
        return f"./results/{stem}.nc"

    def _update_netcdf_output_suggestion(force=False):
        """
        Update export target suggestion only when it is safe:
        - export mode is netcdf
        - current field is empty OR still equals previous auto suggestion
        - or force=True
        """
        if export_mode_w.value != "netcdf":
            return

        new_suggestion = _auto_netcdf_suggestion_from_polygon()
        current = (export_target_w.value or "").strip()
        prev_auto = state.get("last_auto_netcdf_suggestion")

        should_replace = force or (current == "") or (prev_auto is not None and current == prev_auto)

        if should_replace:
            export_target_w.value = new_suggestion

        state["last_auto_netcdf_suggestion"] = new_suggestion

    # -------------------------------------------------------------------------
    # Load and prepare missions metadata
    # -------------------------------------------------------------------------
    df = missions_func().copy()

    # Ignore disabled DEM mission for now
    df = df[df["name"] != "cop_dem_glo_30"].reset_index(drop=True)

    if df.empty:
        raise ValueError("No missions available after filtering.")

    mission_meta = {}
    for _, row in df.iterrows():
        mission_meta[row["name"]] = row.to_dict()

    ordered_names = df["name"].tolist()
    mission_options = [(_pretty_mission_label(name), name) for name in ordered_names]

    # -------------------------------------------------------------------------
    # Widgets (Basic)
    # -------------------------------------------------------------------------
    mission_dd = widgets.Dropdown(
        options=mission_options,
        value=mission_options[0][1],
        description="Mission:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    resolution_w = widgets.IntText(
        value=10,
        description="Resolution:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    polygon_w = widgets.Text(
        value="",
        description="Polygon:",
        placeholder="./polygons/test.gpkg or [xmin, ymin, xmax, ymax]",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    daterange_mode_w = widgets.Dropdown(
        options=[
            ("Standard (single window)", "standard"),
            ("Seasonal (repeat across years)", "seasonal"),
            ("Seasonal + year control", "seasonal_years"),
        ],
        value="standard",
        description="Date Range Mode:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    daterange_w = widgets.Text(
        value="",
        description="Daterange:",
        placeholder=_daterange_mode_placeholder("standard"),
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    bands_w = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Bands:",
        rows=8,
        layout=widgets.Layout(width="100%", height="220px"),
        style={"description_width": "120px"},
    )

    indices_w = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Indices:",
        rows=8,
        layout=widgets.Layout(width="100%", height="220px"),
        style={"description_width": "120px"},
    )

    bands_all_btn = widgets.Button(description="All bands", layout=widgets.Layout(width="110px"))
    bands_none_btn = widgets.Button(description="Clear bands", layout=widgets.Layout(width="110px"))
    indices_all_btn = widgets.Button(description="All indices", layout=widgets.Layout(width="120px"))
    indices_none_btn = widgets.Button(description="Clear indices", layout=widgets.Layout(width="120px"))

    # -------------------------------------------------------------------------
    # Widgets (Advanced)
    # -------------------------------------------------------------------------
    clip_raster_w = widgets.Dropdown(
        options=[("False", False), ("True", True)],
        value=False,
        description="Clip raster:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    max_cc_w = widgets.IntText(
        value=100,
        description="Max CC:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    cloud_masking_w = widgets.Dropdown(
        options=[("False", False), ("True", True)],
        value=False,
        description="Cloud masking:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    stats_w = widgets.SelectMultiple(
        options=[],
        value=(),
        description="Stats:",
        rows=8,
        layout=widgets.Layout(width="100%", height="220px"),
        style={"description_width": "120px"},
    )

    stats_all_btn = widgets.Button(description="All stats", layout=widgets.Layout(width="110px"))
    stats_none_btn = widgets.Button(description="Clear stats", layout=widgets.Layout(width="110px"))

    aggregator_w = widgets.Dropdown(
        options=[("None", None)],
        value=None,
        description="Aggregator:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    # -------------------------------------------------------------------------
    # Widgets (Export Options)
    # -------------------------------------------------------------------------
    export_mode_w = widgets.Dropdown(
        options=[
            ("Quick Result, no Export", "lazy"),
            ("NetCDF", "netcdf"),
            ("Cloud Optimized Geotiffs (select folder)", "cogs"),
        ],
        value="lazy",
        description="Export mode:",
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    export_target_w = widgets.Text(
        value="",
        description="Output:",
        placeholder="Disabled (Return Lazy array selected)",
        disabled=True,
        layout=widgets.Layout(width="100%"),
        style={"description_width": "120px"},
    )

    # -------------------------------------------------------------------------
    # Output / debug + execution
    # -------------------------------------------------------------------------
    params_out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "6px"})
    result_out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "6px"})
    status_out = widgets.Output(layout={"border": "1px solid #ddd", "padding": "6px"})

    show_params_btn = widgets.Button(description="Show current parameters", icon="list")
    generate_btn = widgets.Button(
        description="Generate data cube",
        button_style="success",
        icon="play",
        layout=widgets.Layout(width="190px"),
    )
    export_result_btn = widgets.Button(
        description="Export current result",
        button_style="primary",
        icon="save",
        layout=widgets.Layout(width="190px"),
        disabled=True,
    )

    state = {
        "result": None,
        "last_call_params": None,
        "last_export_info": None,
        "last_auto_netcdf_suggestion": None,
    }


    # -------------------------------------------------------------------------
    # Parsing / validation helpers
    # -------------------------------------------------------------------------
    def _parse_polygon_input(text: str):
        s = (text or "").strip()
        if s == "":
            return None

        if s.startswith("[") or s.startswith("("):
            try:
                obj = ast.literal_eval(s)
            except Exception as e:
                raise ValueError(f"Polygon bbox could not be parsed: {e}")

            if not isinstance(obj, (list, tuple)) or len(obj) != 4:
                raise ValueError("Polygon bbox must be a list/tuple of 4 values: [xmin, ymin, xmax, ymax]")

            try:
                vals = [float(v) for v in obj]
            except Exception:
                raise ValueError("Polygon bbox values must be numeric")

            return vals

        return s  # path string

    def _is_str_list_len2(obj):
        return isinstance(obj, (list, tuple)) and len(obj) == 2 and all(isinstance(x, str) for x in obj)

    def _validate_date_string(s: str, pattern: str, label: str):
        if not re.match(pattern, s):
            raise ValueError(f"Invalid {label}: '{s}'")

    def _parse_daterange_input(mode: str, text: str):
        s = (text or "").strip()
        if s == "":
            return None

        try:
            obj = ast.literal_eval(s)
        except Exception as e:
            raise ValueError(f"Daterange could not be parsed. Please use Python-style list/dict syntax. ({e})")

        if mode == "standard":
            if not _is_str_list_len2(obj):
                raise ValueError('Standard mode expects: ["YYYY-MM-DD", "YYYY-MM-DD"]')
            for d in obj:
                _validate_date_string(d, r"^\d{4}-\d{2}-\d{2}$", "date (YYYY-MM-DD)")
            return list(obj)

        elif mode == "seasonal":
            if not _is_str_list_len2(obj):
                raise ValueError('Seasonal mode expects: ["MM-DD", "MM-DD"]')
            for d in obj:
                _validate_date_string(d, r"^\d{2}-\d{2}$", "season date (MM-DD)")
            return list(obj)

        elif mode == "seasonal_years":
            if not isinstance(obj, dict):
                raise ValueError(
                    'Seasonal + year control expects a dict, e.g. '
                    '{"season": ["04-01", "10-31"], "years": [2019, 2020]}'
                )

            if "season" not in obj or "years" not in obj:
                raise ValueError('Seasonal + year control requires keys: "season" and "years"')

            season = obj["season"]
            years = obj["years"]

            if not _is_str_list_len2(season):
                raise ValueError('"season" must be ["MM-DD", "MM-DD"]')
            for d in season:
                _validate_date_string(d, r"^\d{2}-\d{2}$", 'season date (MM-DD)')

            valid_years = False
            if years == "all":
                valid_years = True
            elif isinstance(years, str) and re.match(r"^\d{4}-\d{4}$", years):
                valid_years = True
            elif isinstance(years, (list, tuple)) and all(isinstance(y, int) for y in years):
                valid_years = True

            if not valid_years:
                raise ValueError(
                    '"years" must be one of: "all", "YYYY-YYYY", or a list of years like [2019, 2020, 2021]'
                )

            return {"season": list(season), "years": years}

        else:
            raise ValueError(f"Unknown Date Range Mode: {mode}")

    def _is_lazy_xarray(obj):
        try:
            chunks = getattr(obj, "chunks", None)
            return chunks is not None
        except Exception:
            return False

    def _human_readable_bytes(n):
        if n is None:
            return "unknown"
        n = float(n)
        units = ["B", "KB", "MB", "GB", "TB", "PB"]
        i = 0
        while n >= 1024 and i < len(units) - 1:
            n /= 1024.0
            i += 1
        return f"{n:.2f} {units[i]}"

    def _estimated_data_size_bytes(obj):
        """
        Estimated uncompressed data size (shape * dtype), no compute triggered.
        This is NOT the final exported file size on disk.
        """
        try:
            if isinstance(obj, xr.DataArray):
                return int(getattr(obj, "nbytes", 0))

            elif isinstance(obj, xr.Dataset):
                total = 0
                for _, da in obj.data_vars.items():
                    try:
                        total += int(getattr(da, "nbytes", 0))
                    except Exception:
                        pass
                return total

            return None
        except Exception:
            return None

    def _prepare_get_stac_layers_params():
        mission = mission_dd.value
        polygon = _parse_polygon_input(polygon_w.value)
        daterange = _parse_daterange_input(daterange_mode_w.value, daterange_w.value)

        resolution = None if resolution_w.disabled else int(resolution_w.value)
        max_cc = None if max_cc_w.disabled else int(max_cc_w.value)

        bands = list(bands_w.value) if len(bands_w.value) > 0 else None
        indices = list(indices_w.value) if len(indices_w.value) > 0 else None
        stats = list(stats_w.value) if len(stats_w.value) > 0 else None

        clip_raster = clip_raster_w.value
        cloud_masking = cloud_masking_w.value
        aggregator = aggregator_w.value

        export_mode = export_mode_w.value
        export_target = (export_target_w.value or "").strip() or None

        # Direct export only for NetCDF mode during generation
        output_for_get_stac = export_target if (export_mode == "netcdf" and export_target) else None

        params = {
            "mission": mission,
            "polygon": polygon,
            "resolution": resolution,
            "daterange": daterange,
            "bands": bands,
            "max_cc": max_cc,
            "clip_raster": clip_raster,
            "cloud_masking": cloud_masking,
            "indices": indices,
            "output": output_for_get_stac,
            "aggregator": aggregator,
            "stats": stats,
            "q": True,
        }

        return params, export_mode, export_target

    def _pick_dataarray_for_cog_export(result_obj):
        if isinstance(result_obj, xr.DataArray):
            da = result_obj
        elif isinstance(result_obj, xr.Dataset):
            if "Spectral_Temporal_Stack" in result_obj.data_vars:
                da = result_obj["Spectral_Temporal_Stack"]
            elif len(result_obj.data_vars) == 1:
                only_name = list(result_obj.data_vars)[0]
                da = result_obj[only_name]
            else:
                raise ValueError(
                    "COG export currently needs a single stack DataArray. "
                    "This result is a Dataset with multiple variables (likely stats outputs). "
                    "Please export as NetCDF or generate without stats."
                )
        else:
            raise TypeError(f"Unsupported result type for export: {type(result_obj)}")

        if "band" not in da.dims:
            raise ValueError(f"COG export requires a 'band' dimension. Found dims: {da.dims}")

        return da

    def _export_current_result(export_mode: str, export_target: str):
        if state["result"] is None:
            raise ValueError("No generated result is available to export yet.")

        if export_mode == "lazy":
            raise ValueError("Please change Export mode to NetCDF or COGs before exporting.")

        if not export_target:
            raise ValueError("Please provide Output file / folder before exporting.")

        obj = state["result"]

        if export_mode == "netcdf":
            target = export_target
            if not target.lower().endswith(".nc"):
                target = target + ".nc"
                export_target_w.value = target

            Path(target).parent.mkdir(parents=True, exist_ok=True)

            if isinstance(obj, xr.DataArray):
                export_stac(stac=obj, output=target, var_name=(obj.name or "Spectral_Temporal_Stack"))
            elif isinstance(obj, xr.Dataset):
                export_stac(stac=obj, output=target)
            else:
                raise TypeError(f"Unsupported result type for NetCDF export: {type(obj)}")

            return {"mode": "netcdf", "target": target}

        elif export_mode == "cogs":
            Path(export_target).mkdir(parents=True, exist_ok=True)
            da = _pick_dataarray_for_cog_export(obj)
            export_to_cogs(stac=da, output_dir=export_target, prefix="", dtype="float32")
            return {"mode": "cogs", "target": export_target}

        else:
            raise ValueError(f"Unsupported export mode: {export_mode}")

    # -------------------------------------------------------------------------
    # Dynamic updates
    # -------------------------------------------------------------------------
    def _apply_export_mode_defaults():
        mode = export_mode_w.value
        current = (export_target_w.value or "").strip()

        if mode == "lazy":
            export_target_w.description = "Output:"
            export_target_w.disabled = True
            export_target_w.placeholder = "Disabled (Return Lazy Array selected)"
            export_target_w.value = ""

        elif mode == "netcdf":
            export_target_w.disabled = False
            export_target_w.description = "Export file:"
            export_target_w.placeholder = "./results/test.nc"

            # If switching from cogs default folder, clear first
            if current in ["./results/cogs", "results/cogs", r"results\cogs"]:
                export_target_w.value = ""

            # Auto-suggest from polygon path (only if safe)
            _update_netcdf_output_suggestion()

        elif mode == "cogs":
            export_target_w.disabled = False
            export_target_w.description = "Export dir:"
            export_target_w.placeholder = "./results/cogs"
            if current == "":
                export_target_w.value = "./results/cogs"

    def _apply_aggregator_stats_logic(*_):
        """
        aggregator != None disables stats (per your docs).
        """
        agg_selected = aggregator_w.value is not None
        meta = mission_meta[mission_dd.value]
        stats_supported = len(_to_list_or_empty(meta.get("stats"))) > 0

        stats_disabled = agg_selected or (not stats_supported)

        stats_w.disabled = stats_disabled
        stats_all_btn.disabled = stats_disabled
        stats_none_btn.disabled = stats_disabled

    def _update_daterange_placeholder(*_):
        daterange_w.placeholder = _daterange_mode_placeholder(daterange_mode_w.value)

    def _update_from_mission(*_):
        m_name = mission_dd.value
        meta = mission_meta[m_name]

        # Resolution
        if _is_supported(meta.get("default_resolution")):
            try:
                resolution_w.value = int(meta["default_resolution"])
            except Exception:
                pass
            resolution_w.disabled = False
        else:
            resolution_w.value = 0
            resolution_w.disabled = True

        # Bands
        bands = _to_list_or_empty(meta.get("bands"))
        bands_w.options = _band_options_with_resolution(m_name, bands)
        bands_w.value = ()
        bands_w.disabled = len(bands) == 0
        bands_all_btn.disabled = len(bands) == 0
        bands_none_btn.disabled = len(bands) == 0

        # Indices
        indices = _to_list_or_empty(meta.get("indices"))
        indices_w.options = _index_options_with_fullname(m_name, indices)
        indices_w.value = ()
        indices_w.disabled = len(indices) == 0
        indices_all_btn.disabled = len(indices) == 0
        indices_none_btn.disabled = len(indices) == 0

        # Clip raster
        clip_cfg = _bool_dropdown_from_metadata(meta.get("clip_raster"), default=False)
        clip_raster_w.options = clip_cfg["options"]
        clip_raster_w.value = clip_cfg["value"]
        clip_raster_w.disabled = clip_cfg["disabled"]

        # Cloud masking
        cm_meta = meta.get("cloud_masking")
        if cm_meta is False:
            cloud_masking_w.options = [("Not available", None)]
            cloud_masking_w.value = None
            cloud_masking_w.disabled = True
        else:
            cm_cfg = _bool_dropdown_from_metadata(cm_meta, default=False)
            cloud_masking_w.options = cm_cfg["options"]
            cloud_masking_w.value = cm_cfg["value"]
            cloud_masking_w.disabled = cm_cfg["disabled"]

        # Max CC
        max_cc_meta = meta.get("max_cc")
        if max_cc_meta is False:
            max_cc_w.value = 0
            max_cc_w.disabled = True
        else:
            try:
                max_cc_w.value = int(max_cc_meta)
            except Exception:
                max_cc_w.value = 100
            max_cc_w.disabled = False

        # Stats
        stats_list = _to_list_or_empty(meta.get("stats"))
        stats_w.options = stats_list
        stats_w.value = ()
        stats_w.disabled = len(stats_list) == 0
        stats_all_btn.disabled = len(stats_list) == 0
        stats_none_btn.disabled = len(stats_list) == 0

        # Aggregator
        agg_list = _to_list_or_empty(meta.get("aggregator"))
        agg_options = [("None", None)] + [(str(x), x) for x in agg_list]
        aggregator_w.options = agg_options
        aggregator_w.value = None
        aggregator_w.disabled = len(agg_list) == 0

        _apply_export_mode_defaults()
        _apply_aggregator_stats_logic()
        _update_daterange_placeholder()

    def _show_current_params(_):
        with params_out:
            clear_output()

            export_mode = export_mode_w.value
            export_target = None if export_target_w.disabled else (export_target_w.value.strip() or None)

            get_stac_output = export_target if (export_mode == "netcdf" and export_target) else None
            deferred_cog_export = export_target if (export_mode == "cogs" and export_target) else None

            params = {
                "mission": mission_dd.value,
                "polygon": polygon_w.value.strip() or None,
                "resolution": None if resolution_w.disabled else int(resolution_w.value),
                "daterange_mode": daterange_mode_w.value,
                "daterange": daterange_w.value.strip() or None,  # raw text
                "bands": list(bands_w.value) if len(bands_w.value) > 0 else None,
                "indices": list(indices_w.value) if len(indices_w.value) > 0 else None,
                "clip_raster": clip_raster_w.value,
                "max_cc": None if max_cc_w.disabled else int(max_cc_w.value),
                "cloud_masking": cloud_masking_w.value,
                "stats": list(stats_w.value) if len(stats_w.value) > 0 else None,
                "aggregator": aggregator_w.value,
                "export_mode": export_mode,
                "export_target": export_target,
                "get_stac_layers(output=...)": get_stac_output,
                "deferred_cog_export_target": deferred_cog_export,
            }

            print("Current UI parameters (raw / preview mapping):")
            for k, v in params.items():
                print(f"- {k}: {v}")

    # -------------------------------------------------------------------------
    # Result / status helpers
    # -------------------------------------------------------------------------
    def _show_status(msg: str, clear_first=True):
        with status_out:
            if clear_first:
                clear_output()
            print(msg)

    def _show_result_summary(obj):
        with result_out:
            clear_output()

            est_bytes = _estimated_data_size_bytes(obj)
            print(f"Estimated data size (uncompressed): {_human_readable_bytes(est_bytes)}")
            print("(Final exported file size may be smaller/larger depending on format, compression, and metadata.)\n")

            display(obj)


    # -------------------------------------------------------------------------
    # Button callbacks
    # -------------------------------------------------------------------------
    def _select_all_bands(_):
        values = []
        for opt in bands_w.options:
            if isinstance(opt, tuple) and len(opt) == 2:
                values.append(opt[1])
            else:
                values.append(opt)
        bands_w.value = tuple(values)

    def _clear_bands(_):
        bands_w.value = ()

    def _select_all_indices(_):
        values = []
        for opt in indices_w.options:
            if isinstance(opt, tuple) and len(opt) == 2:
                values.append(opt[1])
            else:
                values.append(opt)
        indices_w.value = tuple(values)

    def _clear_indices(_):
        indices_w.value = ()

    def _select_all_stats(_):
        if not stats_w.disabled:
            stats_w.value = tuple(stats_w.options)

    def _clear_stats(_):
        stats_w.value = ()

    def _on_generate_clicked(_):
        with result_out:
            clear_output()

        try:
            params, export_mode, export_target = _prepare_get_stac_layers_params()
            state["last_call_params"] = params

            with status_out:
                clear_output()
                print("Generating data cube...")

                # Ensure parent directory exists for direct NetCDF export
                if params["output"] is not None:
                    Path(params["output"]).parent.mkdir(parents=True, exist_ok=True)

                # If get_stac_layers(output=...) calls export_stac(), ProgressBar prints here
                result = get_stac_layers(**params)

                state["result"] = result
                export_result_btn.disabled = False

                # Auto export only if COG mode + target (NetCDF direct export happens via get_stac_layers)
                if export_mode == "cogs" and export_target:
                    print("Generation finished. Exporting current result to COGs...")
                    info = _export_current_result(export_mode, export_target)
                    state["last_export_info"] = info
                    print(f"✅ Data cube generation + COG export finished: {info['target']}")

                elif export_mode == "netcdf" and export_target:
                    state["last_export_info"] = {"mode": "netcdf", "target": export_target, "via": "get_stac_layers"}
                    # export_stac already prints the path -> avoid duplicate path line here
                    print("✅ Data cube generation finished.")

                else:
                    print(
                        "✅ Data cube generation finished. Result stored in memory (lazy if supported). "
                        "Inspect it, then change Export mode if you want to export."
                    )

            # IMPORTANT: show object preview in Result panel (outside status_out context)
            _show_result_summary(state["result"])

        except Exception as e:
            _show_status(f"❌ {type(e).__name__}: {e}")



    def _on_export_result_clicked(_):
        try:
            export_mode = export_mode_w.value
            export_target = None if export_target_w.disabled else ((export_target_w.value or "").strip() or None)

            with status_out:
                clear_output()
                print("Exporting current result...")

                info = _export_current_result(export_mode, export_target)
                state["last_export_info"] = info

                # export_stac() already prints "Export is done: ..."
                # Avoid duplicate final line for NetCDF
                if info.get("mode") != "netcdf":
                    print(f"✅ Export finished: {info['target']}")

        except Exception as e:
            _show_status(f"❌ {type(e).__name__}: {e}")



    # Wire button clicks
    bands_all_btn.on_click(_select_all_bands)
    bands_none_btn.on_click(_clear_bands)
    indices_all_btn.on_click(_select_all_indices)
    indices_none_btn.on_click(_clear_indices)
    stats_all_btn.on_click(_select_all_stats)
    stats_none_btn.on_click(_clear_stats)

    show_params_btn.on_click(_show_current_params)
    generate_btn.on_click(_on_generate_clicked)
    export_result_btn.on_click(_on_export_result_clicked)

    # Observe dynamic changes
    mission_dd.observe(_update_from_mission, names="value")
    aggregator_w.observe(_apply_aggregator_stats_logic, names="value")
    export_mode_w.observe(lambda change: _apply_export_mode_defaults(), names="value")
    daterange_mode_w.observe(_update_daterange_placeholder, names="value")
    polygon_w.observe(lambda change: _update_netcdf_output_suggestion(), names="value")


    # -------------------------------------------------------------------------
    # Layout
    # -------------------------------------------------------------------------
    FORM_WIDTH = "96%"
    FORM_MAX_WIDTH = "950px"

    header = widgets.HTML("<h3 style='margin:0;'>stac2cube Data Cube Generator</h3>")

    bands_box = widgets.VBox([
        _stacked_field(bands_w, "Bands"),
        widgets.HBox([bands_all_btn, bands_none_btn], layout=widgets.Layout(gap="6px"))
    ])

    indices_box = widgets.VBox([
        _stacked_field(indices_w, "Indices"),
        widgets.HBox([indices_all_btn, indices_none_btn], layout=widgets.Layout(gap="6px"))
    ])

    stats_box = widgets.VBox([
        _with_help_left(stats_w, "stats", label_text="Stats"),
        widgets.HBox([stats_all_btn, stats_none_btn], layout=widgets.Layout(gap="6px"))
    ])

    basic_box = widgets.VBox([
        widgets.HTML("<b>Basic Parameters</b>"),
        _stacked_field(mission_dd, "Mission"),
        _stacked_field(resolution_w, "Resolution"),
        _with_help_left(polygon_w, "polygon", label_text="Polygon"),
        _with_help_left(daterange_mode_w, "daterange_mode", label_text="Date Range Mode"),
        _stacked_field(daterange_w, "Daterange"),
        bands_box,
        indices_box,
    ], layout=widgets.Layout(width="100%", gap="6px"))

    advanced_box = widgets.VBox([
        widgets.HTML("<b>Advanced Parameters</b>"),
        _with_help_left(clip_raster_w, "clip_raster", label_text="Clip raster"),
        _with_help_left(max_cc_w, "max_cc", label_text="Max CC"),
        _with_help_left(cloud_masking_w, "cloud_masking", label_text="Cloud masking"),
        stats_box,
        _with_help_left(aggregator_w, "aggregator", label_text="Aggregator"),
    ], layout=widgets.Layout(width="100%", gap="6px"))

    export_box = widgets.VBox([
        widgets.HTML("<b>Export Options</b>"),
        _stacked_field(export_mode_w, "Export mode"),
        _with_help_left(export_target_w, "output", label_text="Output"),
    ], layout=widgets.Layout(width="100%", gap="6px"))

    advanced_acc = widgets.Accordion(children=[advanced_box], selected_index=None)
    advanced_acc.set_title(0, "Advanced parameters")
    advanced_acc.layout = widgets.Layout(width="100%")

    export_acc = widgets.Accordion(children=[export_box], selected_index=None)
    export_acc.set_title(0, "Export Options")
    export_acc.layout = widgets.Layout(width="100%")

    action_row = widgets.HBox(
        [generate_btn, export_result_btn, show_params_btn],
        layout=widgets.Layout(gap="8px", flex_flow="row wrap")
    )

    ui = widgets.VBox([
        header,
        basic_box,
        advanced_acc,
        export_acc,
        action_row,
        widgets.HTML("<b>Result</b>"),
        result_out,
        widgets.HTML("<b>Status</b>"),
        status_out,
        widgets.HTML("<b>Current UI parameters (raw / preview mapping)</b>"),
        params_out,
    ], layout=widgets.Layout(
        width=FORM_WIDTH,
        max_width=FORM_MAX_WIDTH,
        margin="0 auto",
        gap="8px",
    ))

    # Initialize once
    _update_from_mission()

    outer = widgets.HBox(
        [ui],
        layout=widgets.Layout(width="100%", justify_content="center")
    )

    display(outer)

    return {
        "ui": ui,
        "outer": outer,
        "mission_meta": mission_meta,
        "state": state,
        "widgets": {
            "mission": mission_dd,
            "resolution": resolution_w,
            "polygon": polygon_w,
            "daterange_mode": daterange_mode_w,
            "daterange": daterange_w,
            "bands": bands_w,
            "indices": indices_w,
            "clip_raster": clip_raster_w,
            "max_cc": max_cc_w,
            "cloud_masking": cloud_masking_w,
            "stats": stats_w,
            "aggregator": aggregator_w,
            "export_mode": export_mode_w,
            "export_target": export_target_w,
            "generate_btn": generate_btn,
            "export_result_btn": export_result_btn,
            "show_params_btn": show_params_btn,
        },
        "outputs": {
            "result": result_out,
            "status": status_out,
            "params": params_out,
        }
    }


In [24]:
gen_form = launch_stac2cube_generator_form(missions_func=missions)
